# 29 - Agent Observability

Observability answers:

> What did the agent actually do?

Useful trace data:
- run ID
- model calls
- tool calls
- tool arguments
- tool results
- latency
- errors
- token/cost data
- retrieved documents
- final outcome

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "")
if CHAT_MODEL:
    print("Configured model:", CHAT_MODEL)
else:
    print("Set OPENAI_CHAT_MODEL for live model examples.")

## 1. Basic structured logging

Use structured records instead of random print statements.

In [ ]:
import logging
import json
import time

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('agent')

def log_event(event, **data):
    logger.info(json.dumps({
        'event': event,
        **data,
    }))

log_event('tool_start', tool='calculator')

## 2. Trace a tool call

Wrap a tool so you can capture duration and success.

In [ ]:
def traced_tool_call(tool_name, fn, **kwargs):
    start = time.perf_counter()
    try:
        result = fn(**kwargs)
        ok = True
        error = None
        return result
    except Exception as exc:
        ok = False
        error = str(exc)
        raise
    finally:
        duration_ms = (time.perf_counter() - start) * 1000
        log_event(
            'tool_end',
            tool=tool_name,
            duration_ms=round(duration_ms, 2),
            ok=ok,
            error=error,
        )

print(traced_tool_call(
    'calculator',
    lambda expression: eval(expression, {'__builtins__': {}}, {}),
    expression='25 * 4'
))

## 3. Trace shape

A useful trace looks like:

```text
run
 ├─ model_call
 ├─ tool_call
 │   └─ tool_result
 ├─ model_call
 └─ final_answer
```

## 4. Metrics

Track:
- success rate
- p50/p95 latency
- tool error rate
- model call count
- tool call count
- token usage
- estimated cost
- retry count
- safety denials

## 5. Correlating logs

Use stable IDs so events from one agent run can be grouped.

In [ ]:
import uuid

run_id = str(uuid.uuid4())

log_event('run_start', run_id=run_id, task='demo')
log_event('run_end', run_id=run_id, success=True)

## 6. Human-readable debugging

When a task fails, you want to answer:
1. Was the model call wrong?
2. Was the tool wrong?
3. Was the tool result wrong?
4. Was the next decision wrong?
5. Was the final answer wrong?

## 7. Privacy and observability

Do not log secrets or sensitive data unnecessarily.

Use:
- redaction
- field allowlists
- access control
- retention policies

## 8. Evaluation

Observability makes evaluation possible because you need traces to score agent behavior.

Practice: save trace events as JSONL and compute mean tool latency.

## Evaluation

Current MCP Python SDK documentation also exposes OpenTelemetry integration for server operations. For agent applications, tracing systems can similarly capture model/tool steps. citeturn983125view0

Improve with:
- structured logs
- trace IDs
- dashboards
- error grouping
- sensitive-data redaction
- retention policies.

## References / current documentation


- LangChain reference: https://reference.langchain.com/python/langchain/overview
- LangGraph reference: https://reference.langchain.com/python/langgraph/overview
- LangGraph persistence: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/persistence.mdx
- LangGraph interrupts: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/interrupts.mdx
- LangChain human-in-the-loop: https://github.com/langchain-ai/docs/blob/main/src/oss/langchain/human-in-the-loop.mdx
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- MCP Python SDK v2: https://py.sdk.modelcontextprotocol.io/
- MCP Python SDK getting started: https://py.sdk.modelcontextprotocol.io/get-started/